# Plot Vegetable Prices by Product
This notebook loads the vegetable prices dataset (category 13) using the repository,
wrangles it into a DataFrame, removes outliers and splits train/test per product,
then plots the concatenated train data with one trace per product using Plotly.

In [10]:
# Setup imports and path
import sys, os
# ensure repo root is on sys.path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
import pandas as pd
from src.fidzulu.db import oracle_engine
from src.fidzulu.repositories.price_repository import PriceRepository
from src.fidzulu.business.price_wrangler import PriceDataWrangler
from src.fidzulu.business.train_test_splitter import TrainTestSplitter
from src.fidzulu.utils.plotting import price_time_series_figure
print('Imports OK')

Imports OK


In [11]:
# Attempt to fetch vegetable prices (category 13) from the DB
raw_data = None
cache_path = os.path.join('notebooks', 'vegetable_prices_cache.json')
try:
    engine = oracle_engine()
    repo = PriceRepository(engine)
    raw_data = repo.get_prices_by_category(cat_id=13)  # Vegetables
    print('Fetched raw_data from DB; keys:', list(raw_data.keys())[:20])
except Exception as e:
    print('DB access failed:', e)
    # fallback: try loading cached JSON if available
    if os.path.exists(cache_path):
        print('Loading cached dataset from', cache_path)
        raw_data = pd.read_json(cache_path, orient='records')
        # notebooks may have saved a list of records; convert to expected dict if necessary
        if isinstance(raw_data, pd.DataFrame):
            print('Cached file is a DataFrame; cannot convert automatically. Please provide raw_data dict JSON.')
            raw_data = None
    else:
        print('No cache available at', cache_path)

if raw_data is None:
    raise RuntimeError('Unable to obtain vegetable prices dataset from DB or cache. Provide DB access or a cached raw_data dict.')

2026-04-09 09:56:41,532 INFO fidzulu.config: Loaded DBConfig: host=localhost, port=1521, service=xepdb1
2026-04-09 09:56:41,532 INFO src.fidzulu.db: {'event': 'engine_create_attempt', 'db_user': 'fidzulu_pythonmluser', 'host': 'localhost', 'port': 1521, 'service_name': 'xepdb1'}
2026-04-09 09:56:41,533 INFO src.fidzulu.db: Creating Oracle engine with DSN (password redacted from log)
2026-04-09 09:56:41,535 INFO src.fidzulu.db: Oracle engine created successfully
2026-04-09 09:56:41,535 INFO src.fidzulu.repositories.price_repository: {'event': 'query_attempt', 'operation': 'get_prices_by_category', 'query': 'prices_by_category', 'params': {'cat_id': 13}}
2026-04-09 09:56:43,582 WARNING src.fidzulu.repositories.price_repository: {'event': 'anomaly', 'message': 'filtered_invalid_price_rows', 'details': {'category': 13, 'skipped_rows': 1}}


Fetched raw_data from DB; keys: ['CategoryID', 105, 106]


In [12]:
# Convert raw_data (dict as returned by PriceRepository) into a DataFrame via the wrangler
wrangler = PriceDataWrangler(raw_data)
df, feedback = wrangler.wrangle()
print('Wrangled DataFrame shape:', df.shape)
print('Wrangler feedback:', feedback)
df.head()

Wrangled DataFrame shape: (52, 4)
Wrangler feedback: {}


,prod_id,base_price,start_date,end_date
0,105,9.95,2022-11-01,2022-11-30
1,105,2.50,2023-01-01,2023-01-31
2,105,2.65,2023-02-01,2023-02-28
3,105,2.40,2023-03-01,2023-03-31
4,105,2.75,2023-04-01,2023-04-30


In [13]:
# Split and remove outliers, obtain concatenated train DataFrame
splitter = TrainTestSplitter(df, test_ratio=0.2)
concatenated_train, train_splits, test_splits = splitter.split_with_median_iqr_concat()
print('Concatenated train shape:', concatenated_train.shape)
list(train_splits.keys())[:10]

[TrainTestSplitter] product 105: train rows before=22, after=21, test rows=5
[TrainTestSplitter] product 106: train rows before=20, after=20, test rows=5
[TrainTestSplitter] processed 2 products, concatenated train shape=(41, 4)
Concatenated train shape: (41, 4)


[105, 106]

In [15]:
# Create and display Plotly figure (one trace per product)
fig = price_time_series_figure(concatenated_train, title='Vegetable Prices - Train Data')
# In Jupyter this will render inline
fig.show()

[plotting] start_date coerced from datetime64[us] to datetime64[us]; base_price coerced from float64 to float64
[plotting] created figure with 2 traces
